In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

In [ ]:
SAVE_DIR = './../../buckets/manifold-dynamics/diagrams'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

In [ ]:
# TUNING LANDSCAPE
# 2d grid
x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)

# manifold with one clear peak + smaller bumps/valleys
Z = (
    2.5 * np.exp(-((X - 0.6)**2 + (Y - 0.2)**2) / 0.25)  # main peak
    + 1.2 * np.exp(-((X + 1.4)**2 + (Y + 1.0)**2) / 0.6)  # secondary hill
    - 1.0 * np.exp(-((X - 1.2)**2 + (Y + 1.6)**2) / 0.4)  # valley
    - 0.6 * np.exp(-((X + 0.2)**2 + (Y - 1.8)**2) / 0.5)  # valley
    + 0.15 * np.sin(2 * X) * np.cos(2 * Y)                # gentle ripples
)

# plot
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, linewidth=0, cmap='magma', antialiased=True)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_axis_off()

# ax.view_init(elev=25, azim=37, roll=0)

out_path = os.path.join(SAVE_DIR, 'manifold.png')
plt.savefig(out_path, dpi=300, transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# TUNING RDM 01

# t0: nearly uniform similarities
rsa_0 = np.array([
    [1.00, 0.55, 0.54, 0.52, 0.51],
    [0.55, 1.00, 0.56, 0.53, 0.52],
    [0.54, 0.56, 1.00, 0.55, 0.54],
    [0.52, 0.53, 0.55, 1.00, 0.56],
    [0.51, 0.52, 0.54, 0.56, 1.00],
])

# t1: weak clustering begins
rsa_1 = np.array([
    [1.00, 0.70, 0.60, 0.40, 0.38],
    [0.70, 1.00, 0.62, 0.38, 0.36],
    [0.60, 0.62, 1.00, 0.45, 0.43],
    [0.40, 0.38, 0.45, 1.00, 0.68],
    [0.38, 0.36, 0.43, 0.68, 1.00],
])

# t2: strong structure (your original matrix)
rsa_2 = np.array([
    [1.00, 0.88, 0.68, 0.18, 0.12],
    [0.88, 1.00, 0.72, 0.15, 0.10],
    [0.68, 0.72, 1.00, 0.35, 0.28],
    [0.18, 0.15, 0.35, 1.00, 0.86],
    [0.12, 0.10, 0.28, 0.86, 1.00],
])

# t3: structure fades back toward uniform
rsa_3 = np.array([
    [1.00, 0.65, 0.60, 0.45, 0.42],
    [0.65, 1.00, 0.62, 0.43, 0.40],
    [0.60, 0.62, 1.00, 0.50, 0.48],
    [0.45, 0.43, 0.50, 1.00, 0.66],
    [0.42, 0.40, 0.48, 0.66, 1.00],
])

rdms = [rsa_0, rsa_1, rsa_2, rsa_3]

In [ ]:
# TUNING RDM 02
customp = sns.color_palette('rocket_r', as_cmap=True)
fig, axes = plt.subplots(1,4, figsize=(12,3))

# create a colorbar axis to the right of the plots
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
for i in range(len(axes)):
    ax = axes[i]
    rdm = rdms[i]

    sns.heatmap(
        rdm,
        square=True,
        vmin=0,
        vmax=1,
        cmap=customp,
        ax=ax,
        cbar=(i==3),
        cbar_ax=cbar_ax if i==3 else None
    )

    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

In [ ]:
# TUNING RDM 03

# indices for upper triangle (no diagonal)
tri_inds = np.triu_indices(rdms[0].shape[0], k=1)

# extract vectors
vecs = [rdm[tri_inds] for rdm in rdms]

# stack into a matrix for plotting
vec_mat = np.column_stack(vecs)   # shape: (10, 4)

fig, axes = plt.subplots(4, 1, figsize=(3,3))
for i, ax in enumerate(axes):
    v = vecs[i][None, :]   # make it a column (10x1)

    sns.heatmap(
        v,
        cmap=customp,
        vmin=0,
        vmax=1,
        square=True,
        xticklabels=False,
        yticklabels=False,
        cbar=False,
        ax=ax
    )

plt.tight_layout()
plt.show()

In [ ]:
# TUNING RDM 04
# extract and stack vectors: shape = (4, 10)
vec_mat = np.stack([rdm[tri_inds] for rdm in rdms], axis=0)

# RSA across RDMs: correlation between vectorized RDMs
rsa_rdm = np.corrcoef(vec_mat)

customp = sns.color_palette('rocket_r', as_cmap=True)

plt.figure(figsize=(4, 4))
sns.heatmap(
    rsa_rdm,
    cmap=customp,
    vmin=0.99,
    # vmax=1,
    square=True,
    xticklabels=[],
    yticklabels=[],
    cbar=False
)
plt.show()

In [ ]:
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')

# canonical axes
axes = {
    "x": (1, 0, 0),
    "y": (0, 1, 0),
    "z": (0, 0, 1),
}

# 4th axis: close to x, slightly tilted in y (so it forms a plane with x)
extra_axis = (1, 0.3, 0)  # tweak 0.3 → smaller = closer to x

# draw canonical axes
for name, vec in axes.items():
    ax.quiver(0, 0, 0, *vec, length=1, normalize=True)
    ax.text(*(1.1 * v for v in vec), name)

# draw 4th axis
ax.quiver(0, 0, 0, *extra_axis, length=1, normalize=True)
ax.text(*(1.1 * v for v in extra_axis), "x'")

# draw the plane spanned by x and x'
import numpy as np
u = np.linspace(0, 1, 10)
v = np.linspace(0, 1, 10)
U, V = np.meshgrid(u, v)

# plane = span{x, x'}
X = U * 1 + V * extra_axis[0]
Y = U * 0 + V * extra_axis[1]
Z = U * 0 + V * extra_axis[2]

ax.plot_surface(X, Y, Z, alpha=0.2)

# formatting
ax.set_xlim([0,1])
ax.set_ylim([0,1])
ax.set_zlim([0,1])
ax.set_box_aspect([1,1,1])
ax.set_title("3 canonical axes + nearby 4th axis forming a plane with x")

plt.show()